# Odisha – KALIA beneficiary list scraper

Iterates District → Block → Gram Panchayat on the [KALIA portal](https://kaliaportal.odisha.gov.in/Beneficiarylist.aspx) and triggers the beneficiary PDF downloads. A District/Block/GP index CSV is written per block to `data/odisha/`.

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.common import exceptions
from selenium.webdriver.support.select import Select
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
import pickle

In [ ]:
from selenium.webdriver.chrome.service import Service

In [ ]:
import time

In [ ]:
import os

In [ ]:
import pandas as pd

In [ ]:
url ='https://kaliaportal.odisha.gov.in/Beneficiarylist.aspx'

In [ ]:
base_loc = os.path.join(os.getcwd(), 'data', 'odisha')
os.makedirs(base_loc, exist_ok=True)

In [ ]:
options = Options()
options.add_argument("--headless=new")
options.add_argument("--window-size=1920,1200")

In [ ]:
# Uses Selenium Manager (Selenium >= 4.6) to locate ChromeDriver automatically.
# To use a specific driver instead, set the CHROMEDRIVER_PATH environment variable.
CHROMEDRIVER_PATH = os.environ.get("CHROMEDRIVER_PATH")
service = Service(CHROMEDRIVER_PATH) if CHROMEDRIVER_PATH else Service()
driver = webdriver.Chrome(service=service, options=options)

In [ ]:
# driver = webdriver.Chrome()
driver.get(url)
cookies = driver.get_cookies()
actions = ActionChains(driver)

In [ ]:
wait = WebDriverWait(driver, 10)

In [ ]:
def h1_driver(driver):
    h1 = driver.find_element(By.NAME, 'ddlDistrict')
    dist = h1.find_elements(By.TAG_NAME, "option")
    
    dist_val=[]
    for d in dist:
        dist_val.append(d.get_attribute("value"))
    
    for d in dist_val:
        
        district=d
        if d=='--Select--':
            continue
        if not d=='--Select--':
            print(len(dist),d)
            h2_driver(district,driver,d)
    return 

In [ ]:
def h2_driver(district,driver,k):
    #k.click()
    e1=(driver.find_element(By.NAME,'ddlDistrict'))
    s_e1=Select(e1)
    s_e1.select_by_value(district)
    #e1.click()
    #e1.perform()
    h2 = driver.find_element(By.NAME, 'ddlBlock')
    all_options = h2.find_elements(By.TAG_NAME, "option")
    print(len(all_options),district)
    
    block_val=[]
    for d in all_options:
        block_val.append(d.get_attribute("value"))

    for b in block_val:
        block=b
        if b=='--Select--':
            continue
        if not b=='--Select--':
            h3_driver(block,district,driver,d)
            print('Block',len(all_options))


In [ ]:
def h3_driver(block,district,driver,d):
    data=pd.DataFrame()
    dist=[]
    blk=[]
    vill=[]
    e1=(driver.find_element(By.NAME,'ddlDistrict'))
    s_e1=Select(e1)
    s_e1.select_by_value(district)
    
    
    e2=(driver.find_element(By.NAME,'ddlBlock'))
    s_e2=Select(e2)
    s_e2.select_by_value(block)
    
    h3=driver.find_element(By.NAME, 'ddlGP')
    all_gp = h3.find_elements(By.TAG_NAME, "option")
    for d in all_gp:
        gp=d.get_attribute("value")
        if d.get_attribute("value")=='--Select--':
            continue
        if not d.get_attribute("value")=='--Select--':
            data_download(district,block,gp,driver)
            dist.append(district)
            blk.append(block)
            vill.append(gp)
      
    fn=os.path.join(base_loc,str(district)+'-'+str(block)+'.csv')
    data=pd.DataFrame({'District':dist,'Block':blk,'Village':vill})
    data.to_csv(fn)

In [ ]:
def data_download(district,block,gp,driver):
    
    e1=(driver.find_element(By.NAME,'ddlDistrict'))
    s_e1=Select(e1)
    s_e1.select_by_value(district)
    
    
    e2=(driver.find_element(By.NAME,'ddlBlock'))
    s_e2=Select(e2)
    s_e2.select_by_value(block)
    
    e3=(driver.find_element(By.NAME,'ddlGP'))
    s_e3=Select(e3)
    s_e3.select_by_value(gp)
    
    driver.find_element(By.ID,"btnView").click()
    driver.find_element(By.NAME, 'SMFPDF').click()
    driver.find_element(By.NAME, 'LAHPDF').click()
    return

In [ ]:
h1_driver(driver)